# Generation-Scoped Incremental Code Overlay Indexes

**Status:** Approved architecture; written specification awaiting review  
**Design epic:** `bd-20iv`  
**Source optimization:** `sol_24852081d34c479b` (Z3 Optimize 4.16.0, lexicographic, complete)  
**Formal profiles:** `relational_lia@1`, `state_invariant_lia@1`

This specification replaces per-request overlay finalization with one immutable, certified overlay generation whose changed-path indexes are incrementally updated and atomically published. It extends the existing fsmonitor-gated, OID-addressed snapshot and delta caches; it does not replace their exact fallback behavior.

## Problem and grounded evidence

The existing OID-addressed cache reuses the certified snapshot and extracted delta, but the final overlay view remains request-local:

- `OverlayClient::search_symbols` performs unbounded base search, shadow filtering, delta extension, sorting, deduplication, and limiting on every invocation.
- Caller, callee, selector, and symbol lookup paths repeat overlay remapping or filtering on every invocation.
- `RequestReplayClient` memoizes base graph operations only inside one MCP request.
- `CachedOverlayDelta` stores the extracted delta and shadowed paths, not a materialized query view shared by later requests.
- A dirty Spur worktree probe measured snapshot/change validation at an average **70.894016 ms** across five iterations with 3,073 base files and 364 changed paths. This is before final overlay query processing.

Grounded seams:

- `crates/spur-graph/src/query_client.rs`: `OverlayClient` and graph query implementations.
- `crates/spur-graph/src/mcp/request_cache.rs`: snapshot and delta cache identity/LRU.
- `crates/spur-graph/src/mcp/request_replay.rs`: request-local base replay.
- `crates/spur-graph/src/mcp/mod.rs`: freshness analysis, overlay construction, and request dispatch.

## Architecture decision

Z3 Optimize compared four exact, fail-closed candidates using lexicographic objectives:

1. Minimize worst-case query-time overlay reconstruction stages.
2. Minimize Git-change recomputation scope.
3. Minimize coordination surfaces.

| Candidate | Query overlay stages | Git-change scope | Coordination |
|---|---:|---:|---:|
| Per-request finalization | 4 | none | lowest |
| Generation delta + exact-query cache | 4 for a new query shape | changed-set | medium |
| Full combined graph rebuild | 0 | base-wide | medium |
| Generation incremental combined indexes | **0** | **changed-set** | highest |

The complete optimum was **generation incremental combined indexes** (`sol_24852081d34c479b`). The numbers are normalized structural units, not latency predictions.

The implementation SHALL therefore maintain a certified immutable generation whose secondary indexes structurally apply shadowing, replacement, and edge remapping. Query-time scoring and top-k selection remain normal query work; combining separate base and delta result vectors does not.

In [ ]:
flowchart TD
    SPEC["`@spec OVERLAY-GENERATION-ROUTING
@type Route = enum[reuse, publish, exact_reconcile]
@input monitor_healthy: Bool
@input identity_matches: Bool
@input candidate_certified: Bool
@output status: Route
@requires PRE: true`"]

    REUSE["`@branch REUSE
@when monitor_healthy and identity_matches
@ensures REUSE_STATUS: status = reuse`"]

    PUBLISH["`@branch PUBLISH
@when monitor_healthy and not identity_matches and candidate_certified
@ensures PUBLISH_STATUS: status = publish`"]

    RECONCILE["`@branch RECONCILE
@when not monitor_healthy or (not identity_matches and not candidate_certified)
@ensures RECONCILE_STATUS: status = exact_reconcile`"]

    CHECK["`@verify ROUTE_DETERMINISTIC: prove determinism
@verify ROUTE_COVERAGE: prove partition_coverage
@verify ROUTE_EXCLUSIVE: prove partition_exclusive
@verify ROUTES_REACHABLE: witness each status`"]

    SPEC --> REUSE --> CHECK
    SPEC --> PUBLISH --> CHECK
    SPEC --> RECONCILE --> CHECK

## Generation identity and owned state

A generation is keyed by the complete certified identity already used by the overlay lifecycle:

- canonical worktree identity
- graph content hash
- indexed HEAD OID
- current HEAD OID
- Git index identity
- changed-set fingerprint

`OverlayGeneration` is immutable after publication and shared through `Arc`. It owns or references:

- certified changed paths and file OIDs
- shadowed base paths
- delta symbols and file manifests
- stable-symbol and selector maps
- search-key entries with shadowing/deduplication already applied
- caller and callee adjacency patches
- unresolved-label and old-ID remap data needed by existing semantics

The base Parquet artifact stays immutable. Combined indexes may be persistent/layered structures referencing base records; the design does not require copying the complete base graph. There is no TTL. Reuse ends only when the certified identity changes or exact reconciliation invalidates it.

In [ ]:
flowchart TD
    SPEC["`@spec OVERLAY-GENERATION-UPDATE-SCOPE
@type UpdateScope = enum[reuse, patch_changed, rebuild_base]
@input base_identity_changed: Bool
@input worktree_changed: Bool
@output status: UpdateScope
@requires PRE: true`"]

    REUSE["`@branch REUSE
@when not base_identity_changed and not worktree_changed
@ensures REUSE_STATUS: status = reuse`"]

    PATCH["`@branch PATCH_CHANGED
@when not base_identity_changed and worktree_changed
@ensures PATCH_STATUS: status = patch_changed`"]

    REBUILD["`@branch REBUILD_BASE
@when base_identity_changed
@ensures REBUILD_STATUS: status = rebuild_base`"]

    CHECK["`@verify UPDATE_DETERMINISTIC: prove determinism
@verify UPDATE_COVERAGE: prove partition_coverage
@verify UPDATE_EXCLUSIVE: prove partition_exclusive
@verify UPDATE_STATUSES_REACHABLE: witness each status`"]

    SPEC --> REUSE --> CHECK
    SPEC --> PATCH --> CHECK
    SPEC --> REBUILD --> CHECK

In [ ]:
stateDiagram-v2
    [*] --> Stable
    Stable --> Building: change
    Building --> StableNext: build_complete
    Building --> Reconciling: monitor_failure
    Reconciling --> Reconciled: reconcile_complete

    note right of Stable
      @spec OVERLAY-GENERATION-PUBLICATION
      @type GenerationEvent = enum[change, build_complete, monitor_failure, reconcile_complete]
      @input event: GenerationEvent
      @input candidate_certified: Bool
      @input monitor_healthy: Bool
      @input exact_check_ok: Bool
      @state-var published_certified: Bool
      @requires INITIAL_CERTIFIED: published_certified = true
      @state Stable
      @invariant VISIBLE_CERTIFIED: published_certified = true
    end note

    note right of Building
      @state Building
      @transition START_BUILD
      @from Stable
      @to Building
      @event event = change
      @guard published_certified = true
      @update published_certified' = true
    end note

    note right of StableNext
      @state StableNext
      @transition PUBLISH
      @from Building
      @to StableNext
      @event event = build_complete
      @guard candidate_certified = true
      @update published_certified' = true
    end note

    note right of Reconciling
      @state Reconciling
      @transition FALLBACK
      @from Building
      @to Reconciling
      @event event = monitor_failure
      @guard not monitor_healthy or not candidate_certified
      @update published_certified' = true
    end note

    note right of Reconciled
      @state Reconciled
      @transition RECONCILE
      @from Reconciling
      @to Reconciled
      @event event = reconcile_complete
      @guard exact_check_ok = true
      @update published_certified' = true
      @verify INIT_VISIBLE: prove initiate VISIBLE_CERTIFIED
      @verify PRESERVE_START: prove preserve VISIBLE_CERTIFIED on START_BUILD
      @verify PRESERVE_PUBLISH: prove preserve VISIBLE_CERTIFIED on PUBLISH
      @verify PRESERVE_FALLBACK: prove preserve VISIBLE_CERTIFIED on FALLBACK
      @verify PRESERVE_RECONCILE: prove preserve VISIBLE_CERTIFIED on RECONCILE
    end note

## Query semantics

Every MCP request loads one published generation and keeps that same `Arc` for the complete handler execution, including nested subgraph traversal.

- **Search:** query a generation-scoped combined search index. Shadowed base entries and duplicate stable IDs were resolved during generation construction. The request performs matching, scoring, and top-k selection only.
- **Symbol lookup / selector resolution:** consult generation maps that contain delta replacement and base visibility policy.
- **Callers / callees:** consult generation adjacency maps with changed edges and old-ID remaps already applied.
- **File and manifest lookup:** consult the generation file view.
- **Identity overlay:** continue using the direct base client fast path.

A cache of exact normalized query results may be added inside a generation as a secondary optimization, but correctness and the primary acceptance criteria cannot depend on repeated arguments. Different sequential operations must reuse the same combined view.

## Freshness, concurrency, and fallback

Git monitor notifications are hints, never proof.

1. A Git or filesystem signal marks the canonical worktree generation dirty and coalesces pending changed paths.
2. One singleflight builder performs exact Git reconciliation, computes the next changed set, applies incremental index patches, and certifies the full identity.
3. Publication is atomic. Readers see either the previous certified generation or the complete next certified generation—never partial state.
4. A request whose identity matches the published generation reuses it without reconstruction.
5. Monitor unavailability, overflow, contradictory identity, or a failed candidate build routes to exact reconciliation.
6. Exact-at-call-time semantics are preserved: an uncertain request waits or performs synchronous reconciliation; it does not silently serve stale state.

Old generations may remain alive while pinned by requests. Eviction reuses the existing identity-based cache policy unless measurements and a separate SOLVE model justify a new bound.

## TDD, SOLVE, and performance acceptance

Every implementation task follows:

1. **SOLVE PRE:** reload `sol_24852081d34c479b`, navigate the current rule catalog, and persist any task-specific feasibility/safety model.
2. **RED:** commit a failing behavioral test before production edits.
3. **GREEN:** implement only the tested behavior and verify the focused and crate suites with `scripts/spur-cargo`.
4. **SOLVE POST:** re-encode the landed policy with the same variables and constraints. Unknown or timeout is not success.
5. **POST measurement:** repeat the same fixtures, query shapes, warm/cold classification, and harness-configured repetitions used for PRE.

Required behavioral coverage:

- different sequential graph operations reuse one published generation
- a changed path patches the next generation and invalidates the prior identity
- unchanged paths retain their indexed records
- shadowing and stable-ID deduplication match current overlay semantics
- caller/callee remaps match current overlay semantics
- concurrent readers never observe partial publication
- monitor loss and overflow take the exact fallback path
- clean and identity-overlay paths remain direct

Performance evidence must separately report snapshot certification, generation construction, query execution, total wall time, cache/generation hit state, project scale, and correctness digest. Release requires zero digest mismatches and removal of the four overlay-finalization stages from the warm query path.

## Scope boundaries, migration, and risks

### Proposed implementation boundaries

- **Generation model/store:** cache identity, immutable publication, and singleflight lifecycle in `mcp/request_cache.rs`.
- **Incremental query indexes:** combined symbol/search/adjacency views in `query_client.rs` or a focused new crate-local module.
- **MCP integration:** generation acquisition and exact fallback routing in `mcp/mod.rs`.
- **Evidence:** behavioral counters and PRE/POST scenarios in existing unit tests and `benches/overlay.rs`.

These boundaries become dependency-ordered plan tasks; implementation workers must emit scope drift before touching files outside their assigned set.

### Migration

The feature remains gated by the existing Off/Auto configuration. Auto first attempts the generation path and falls back to the current exact overlay construction on any unsupported or unhealthy condition. Removal of the old path is out of scope until POST measurements and compatibility tests prove the new path.

### Non-goals

- replacing Git's correctness authority with watcher events
- serving silently stale generations
- rebuilding or copying the full base graph after each worktree edit
- changing graph extraction semantics or stable-symbol IDs
- inventing a new cache TTL or capacity without measurement and SOLVE evidence

### Principal risks

- fuzzy-search equivalence when moving deduplication from response vectors into an index
- edge remap equivalence for renamed or deleted symbols
- memory retention by pinned generations
- change storms causing build starvation
- invalidation races between Git certification and publication

Each risk requires a failing regression test or same-shape measurement before its implementation task can close.

## Formal proof evidence

| @spec | Cell | Profile | Obligations | Source hash | IR hash | Report hash |
|---|---|---|---:|---|---|---|
| `OVERLAY-GENERATION-ROUTING` | `…0004` | `relational_lia@1` | 6/6 matched | `7a4807db18a31dc09afc3796ed30fa1f0a6a3bff7c5a291e5b33bca00c3ed97f` | `1e6ff33f5414bc7e9308f1c5d51631b44e58f01ce71fb455947006031976c20d` | `34ba5a4c9412428eda75a8422f586825acc10f9bb38cd9d98347e160d004e4d6` |
| `OVERLAY-GENERATION-UPDATE-SCOPE` | `…0006` | `relational_lia@1` | 6/6 matched | `153721373bc27179a680704d2f89c2251c0dfada092650133f6089d13f9e09a4` | `9c2d195e85df6748166f736ddb9fd787431add58013e539ec9ebe5902d8f5a70` | `7d16d3fd1f7dad715a7d08ae5f3c62b8a1bfe386cf814e4c4f8611d635d0db42` |
| `OVERLAY-GENERATION-PUBLICATION` | `…0007` | `state_invariant_lia@1` | 10/10 matched | `5545daeb806d9280d6f5da3d3fea0d6c7ee23aac843c3c0cdce23511a2e15f3c` | `1729eee95134187333cf2b16c0d4663567409faeffada2a7fcbd3239b39c82c4` | `510bf751e18966459f870b08639553857af997190a069543299678c1baea10b9` |

All mandatory facets are verified and proof-fresh. The independent architecture optimization is persisted as `sol_24852081d34c479b`; NS-Mermaid does not duplicate Optimize because the formal notebook profile intentionally supports hard QF_LIA contracts only.